1. Connect Google Drive and import libraries

In [11]:
from google.colab import drive
drive.mount("/content/drive")

import pandas as pd
import numpy as np
from pathlib import Path

# Notice the trailing space after 'Network Analysis '
BASE = Path("/content/drive/MyDrive/Network Analysis /Data")
RAW = BASE / "raw"
PROCESSED = BASE / "Processed"

PROCESSED.mkdir(parents=True, exist_ok=True)

print("Raw folder exists:", RAW.exists())
print("Files found:", [p.name for p in RAW.glob("*.csv")])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Raw folder exists: True
Files found: ['April 2025.csv', 'May 2025.csv', 'June 2025.csv']


2 . Load and validate the raw datasets

In [13]:

files = {
    4: "April 2025.csv",
    5: "May 2025.csv",
    6: "June 2025.csv"
}

frames = []

for month, filename in files.items():
    df = pd.read_csv(RAW / filename, low_memory=False)

    # Check that each file contains the expected period
    assert (df["YEAR"] == 2025).all()
    assert (df["MONTH"] == month).all()

    df["FL_DATE"] = pd.to_datetime(df["FL_DATE"])

    assert (df["FL_DATE"].dt.year == 2025).all()
    assert (df["FL_DATE"].dt.month == month).all()

    print(f"{filename}: {len(df):,} records")
    frames.append(df)

# All monthly files should have the same columns
assert all(set(f.columns) == set(frames[0].columns)
           for f in frames)

flights = pd.concat(frames, ignore_index=True)

print("\nTotal records:", f"{len(flights):,}")
print("Columns:", len(flights.columns))


/tmp/ipykernel_466/2673407056.py:16: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["FL_DATE"] = pd.to_datetime(df["FL_DATE"])


April 2025.csv: 644,084 records


/tmp/ipykernel_466/2673407056.py:16: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["FL_DATE"] = pd.to_datetime(df["FL_DATE"])


May 2025.csv: 667,586 records


/tmp/ipykernel_466/2673407056.py:16: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["FL_DATE"] = pd.to_datetime(df["FL_DATE"])


June 2025.csv: 674,179 records

Total records: 1,985,849
Columns: 19


Filter the airlines

In [14]:

AIRLINES = ["AA", "DL", "WN"]

flights = flights[
    flights["MKT_UNIQUE_CARRIER"].isin(AIRLINES)
].copy()

print("Selected flight records:", f"{len(flights):,}")

display(
    flights.groupby(["MKT_UNIQUE_CARRIER", "MONTH"])
           .size()
           .unstack(fill_value=0)
)


Selected flight records: 1,292,785


MONTH,4,5,6
MKT_UNIQUE_CARRIER,,,
AA,166972,173676,173558
DL,133985,141809,142996
WN,117622,119819,122348


4. Validate our network construction assumptions

**Question	Decision**


*   What represents a node?	A distinct airport ID
*   When does an edge exist?	At least one scheduled nonstop flight
*   Is the network directed?	Yes, origin → destination
*   What is the edge weight?	Total number of scheduled flights
*   Do we include canceled flights?	Yes, because we study scheduled connectivity
*   Do we use operating or marketing carrier?	Marketing carrier, including regional partners
*   Do we include diverted flights?	Yes, using their originally scheduled destination

Check missing values and flags

In [15]:

essential = [
    "FL_DATE",
    "MKT_UNIQUE_CARRIER",
    "ORIGIN_AIRPORT_ID",
    "DEST_AIRPORT_ID",
    "FLIGHTS"
]

print("Missing essential values:")
print(flights[essential].isna().sum())

print("\nFlight count values:")
print(flights["FLIGHTS"].value_counts(dropna=False))

print("\nCancellation summary:")
display(
    flights.groupby("MKT_UNIQUE_CARRIER")
    .agg(
        records=("FLIGHTS", "size"),
        scheduled_flights=("FLIGHTS", "sum"),
        cancelled=("CANCELLED", "sum"),
        diverted=("DIVERTED", "sum")
    )
)


Missing essential values:
FL_DATE               0
MKT_UNIQUE_CARRIER    0
ORIGIN_AIRPORT_ID     0
DEST_AIRPORT_ID       0
FLIGHTS               0
dtype: int64

Flight count values:
FLIGHTS
1.0    1292785
Name: count, dtype: int64

Cancellation summary:


,records,scheduled_flights,cancelled,diverted
MKT_UNIQUE_CARRIER,,,,
AA,514206,514206.0,11726.0,1743.0
DL,418790,418790.0,4541.0,1141.0
WN,359789,359789.0,1967.0,946.0


Establish the validated records

In [16]:

# Only include meaningful scheduled flight records
clean = flights.dropna(subset=essential).copy()

clean = clean[clean["FLIGHTS"] > 0].copy()

# Remove impossible self-loops if any exist
self_loops = (
    clean["ORIGIN_AIRPORT_ID"]
    == clean["DEST_AIRPORT_ID"]
).sum()

print("Self-loop flight records:", self_loops)

# Do not silently discard self-loops: investigate if found
assert self_loops == 0, "Review origin=destination records"

print("Validated records:", f"{len(clean):,}")


Self-loop flight records: 0
Validated records: 1,292,785


5. Construct the three airline networks

Aggregate scheduled flights into edges

In [17]:

edges = (
    clean.groupby(
        [
            "MKT_UNIQUE_CARRIER",
            "ORIGIN_AIRPORT_ID",
            "DEST_AIRPORT_ID"
        ],
        as_index=False
    )
    .agg(frequency=("FLIGHTS", "sum"))
)

edges = edges.rename(columns={
    "MKT_UNIQUE_CARRIER": "airline",
    "ORIGIN_AIRPORT_ID": "source",
    "DEST_AIRPORT_ID": "target"
})

display(edges.head())

print("\nDirected routes per airline:")
print(edges.groupby("airline").size())


,airline,source,target,frequency
0,AA,10135,11057,182.0
1,AA,10136,11298,439.0
2,AA,10140,11298,653.0
3,AA,10140,12892,234.0
4,AA,10140,13930,178.0



Directed routes per airline:
airline
AA    1808
DL    1506
WN    1792
dtype: int64


Create the airport node table

In [18]:

origin_nodes = clean[
    ["ORIGIN_AIRPORT_ID", "ORIGIN",
     "ORIGIN_CITY_NAME", "ORIGIN_STATE_ABR"]
].copy()

origin_nodes.columns = [
    "airport_id", "code", "city", "state"
]

dest_nodes = clean[
    ["DEST_AIRPORT_ID", "DEST",
     "DEST_CITY_NAME", "DEST_STATE_ABR"]
].copy()

dest_nodes.columns = [
    "airport_id", "code", "city", "state"
]

nodes = pd.concat(
    [origin_nodes, dest_nodes],
    ignore_index=True
).drop_duplicates()

# Verify airport IDs have consistent metadata
conflicts = nodes.groupby("airport_id").agg(
    codes=("code", "nunique"),
    cities=("city", "nunique"),
    states=("state", "nunique")
)

assert (conflicts <= 1).all().all(), (
    "Some airport IDs have conflicting metadata"
)

nodes = (
    nodes.drop_duplicates(subset=["airport_id"])
    .sort_values("airport_id")
    .reset_index(drop=True)
)

print("Unique airport IDs:", len(nodes))
display(nodes.head())


Unique airport IDs: 272


,airport_id,code,city,state
0,10135,ABE,"Allentown/Bethlehem/Easton, PA",PA
1,10136,ABI,"Abilene, TX",TX
2,10140,ABQ,"Albuquerque, NM",NM
3,10141,ABR,"Aberdeen, SD",SD
4,10146,ABY,"Albany, GA",GA


Export six shared CSV files

In [20]:

for airline in AIRLINES:
    airline_edges = (
        edges[edges["airline"] == airline]
        [["source", "target", "frequency"]]
        .copy()
    )

    airport_ids = set(airline_edges["source"]) | set(
        airline_edges["target"]
    )

    airline_nodes = (
        nodes[nodes["airport_id"].isin(airport_ids)]
        .copy()
    )

    airline_edges.to_csv(
        PROCESSED / f"{airline}_edges.csv",
        index=False
    )

    airline_nodes.to_csv(
        PROCESSED / f"{airline}_nodes.csv",
        index=False
    )

    print(
        airline,
        "| airports:", len(airline_nodes),
        "| routes:", len(airline_edges)
    )


AA | airports: 230 | routes: 1808
DL | airports: 212 | routes: 1506
WN | airports: 104 | routes: 1792


6. Build and validate the NetworkX graphs

Construct directed weighted graphs

In [21]:

import networkx as nx

graphs = {}

for airline in AIRLINES:
    airline_nodes = pd.read_csv(
        PROCESSED / f"{airline}_nodes.csv"
    )
    airline_edges = pd.read_csv(
        PROCESSED / f"{airline}_edges.csv"
    )

    G = nx.DiGraph(airline=airline)

    for row in airline_nodes.itertuples(index=False):
        G.add_node(
            int(row.airport_id),
            code=row.code,
            city=row.city,
            state=row.state
        )

    for row in airline_edges.itertuples(index=False):
        G.add_edge(
            int(row.source),
            int(row.target),
            frequency=float(row.frequency)
        )

    graphs[airline] = G

    assert G.number_of_nodes() == len(airline_nodes)
    assert G.number_of_edges() == len(airline_edges)

    print(
        airline,
        "Nodes:", G.number_of_nodes(),
        "Edges:", G.number_of_edges(),
        "Directed:", G.is_directed()
    )


AA Nodes: 230 Edges: 1808 Directed: True
DL Nodes: 212 Edges: 1506 Directed: True
WN Nodes: 104 Edges: 1792 Directed: True


Produce the first network summary table

In [23]:

summary = []

for airline, G in graphs.items():
    summary.append({
        "Airline": airline,
        "Airports": G.number_of_nodes(),
        "Directed routes": G.number_of_edges(),
        "Density": nx.density(G),
        "Total scheduled flights": sum(
            data["frequency"]
            for _, _, data in G.edges(data=True)
        ),
        "Strongly connected components":
            nx.number_strongly_connected_components(G)
    })

summary_df = pd.DataFrame(summary)

display(summary_df.round(4))

summary_df.to_csv(
    PROCESSED / "network_summary.csv",
    index=False
)


,Airline,Airports,Directed routes,Density,Total scheduled flights,Strongly connected components
0,AA,230,1808,0.0343,514206.0,1
1,DL,212,1506,0.0337,418790.0,1
2,WN,104,1792,0.1673,359789.0,1


Final validation before distributing the data

In [24]:

raw_counts = (
    clean.groupby("MKT_UNIQUE_CARRIER")["FLIGHTS"]
    .sum()
    .sort_index()
)

aggregated_counts = (
    edges.groupby("airline")["frequency"]
    .sum()
    .sort_index()
)

aggregated_counts.index.name = "MKT_UNIQUE_CARRIER"

pd.testing.assert_series_equal(
    raw_counts,
    aggregated_counts,
    check_names=False
)

print("PASS: Flight frequencies match the raw data.")

for airline, G in graphs.items():
    assert nx.number_of_selfloops(G) == 0

    assert all(
        data["frequency"] > 0
        for _, _, data in G.edges(data=True)
    )

print("PASS: No self-loops or invalid edge weights.")
print("PASS: Graph construction is complete.")


PASS: Flight frequencies match the raw data.
PASS: No self-loops or invalid edge weights.
PASS: Graph construction is complete.
